# Notebook 01 — Dataset Preparation & Exploratory Data Analysis
### XAI for Music Genre Classification | GTZAN Dataset
---
**Purpose:** Download GTZAN, audit known faults, extract features, and produce HD publication-ready EDA figures.

**Outputs:**
- `../data/gtzan_metadata.csv` — per-clip metadata with fault flags
- `../figures/01_*.png` — HD EDA plots (300 DPI)

**References:**
- Tzanetakis & Cook (2002) — original GTZAN paper
- Sturm (2013) — fault analysis

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

packages = [
    'librosa', 'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scikit-learn', 'tqdm', 'soundfile', 'kaggle'
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Dependencies ready.')

In [ ]:
# ── Cell 2: Imports & global plot config ──────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Publication-grade plot config ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'DejaVu Sans',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.prop_cycle': matplotlib.cycler(color=[
        '#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
        '#00BCD4','#E91E63','#8BC34A','#FF5722','#607D8B'
    ])
})

FIGURES = Path('../figures')
DATA    = Path('../data')
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

GENRE_COLORS = dict(zip(GENRES, plt.rcParams['axes.prop_cycle'].by_key()['color']))

def savefig(name, fig=None, tight=True):
    """Save figure at publication quality."""
    if fig is None:
        fig = plt.gcf()
    if tight:
        fig.tight_layout()
    path = FIGURES / f'{name}.png'
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'  ✓ Saved {path}')
    plt.show()

print('Config loaded. Genre palette set.')

In [ ]:
# ── Cell 3: Download GTZAN ─────────────────────────────────────────────────────
# Option A: Kaggle API (recommended)
# Place kaggle.json in ~/.kaggle/ then run:
#   kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification

# Option B: Manual path — set GTZAN_ROOT to your local extract
GTZAN_ROOT = Path(os.environ.get('GTZAN_ROOT', '../data/gtzan/genres_original'))

if not GTZAN_ROOT.exists():
    print('GTZAN not found. Running Kaggle download...')
    os.makedirs('../data/gtzan', exist_ok=True)
    os.system('kaggle datasets download -d andradaolteanu/gtzan-dataset-music-genre-classification -p ../data/gtzan --unzip')
    # After unzip the path is usually: ../data/gtzan/Data/genres_original
    GTZAN_ROOT = Path('../data/gtzan/Data/genres_original')

print(f'GTZAN root: {GTZAN_ROOT}')
print(f'Exists: {GTZAN_ROOT.exists()}')

In [ ]:
# ── Cell 4: Audit & Metadata extraction ──────────────────────────────────────
# Known GTZAN faults from Sturm (2013):
KNOWN_FAULTS = {
    'blues.00068.wav':     'corrupt',
    'jazz.00054.wav':      'corrupt',
    'jazz.00055.wav':      'duplicate',
    'jazz.00056.wav':      'duplicate',
    'jazz.00057.wav':      'duplicate',
    'jazz.00058.wav':      'duplicate',
    'rock.00081.wav':      'mislabeled',
    'classical.00049.wav': 'mislabeled',
}

records = []
for genre in tqdm(GENRES, desc='Scanning clips'):
    genre_dir = GTZAN_ROOT / genre
    if not genre_dir.exists():
        print(f'  Warning: {genre_dir} not found')
        continue
    for wav in sorted(genre_dir.glob('*.wav')):
        fault = KNOWN_FAULTS.get(wav.name, 'clean')
        rec = {
            'filepath': str(wav),
            'filename': wav.name,
            'genre': genre,
            'fault': fault,
            'file_size_kb': wav.stat().st_size / 1024,
        }
        # Load audio metadata without loading full array
        try:
            duration = librosa.get_duration(path=str(wav))
            rec['duration_s'] = round(duration, 3)
        except Exception as e:
            rec['duration_s'] = None
            rec['fault'] = 'corrupt'
        records.append(rec)

meta = pd.DataFrame(records)
meta.to_csv(DATA / 'gtzan_metadata.csv', index=False)
print(f'\nTotal clips found: {len(meta)}')
print(f'Fault breakdown:\n{meta["fault"].value_counts()}')
meta.head()

In [ ]:
# ── Cell 5: Figure 01 — Dataset composition & fault map ──────────────────────
fig = plt.figure(figsize=(16, 6))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Panel A: clips per genre
ax1 = fig.add_subplot(gs[0])
counts = meta.groupby('genre').size().reindex(GENRES)
bars = ax1.bar(GENRES, counts, color=[GENRE_COLORS[g] for g in GENRES], edgecolor='white', linewidth=0.8)
ax1.set_xticklabels(GENRES, rotation=45, ha='right')
ax1.set_ylabel('Number of clips')
ax1.set_title('A  Clips per Genre')
ax1.set_ylim(0, 115)
for bar, v in zip(bars, counts):
    ax1.text(bar.get_x() + bar.get_width()/2, v + 1, str(v), ha='center', va='bottom', fontsize=9)

# Panel B: duration distribution
ax2 = fig.add_subplot(gs[1])
valid = meta.dropna(subset=['duration_s'])
ax2.hist(valid['duration_s'], bins=30, color='#2196F3', edgecolor='white', linewidth=0.5)
ax2.axvline(30.0, color='#F44336', linestyle='--', linewidth=1.5, label='Expected 30s')
ax2.set_xlabel('Duration (seconds)')
ax2.set_ylabel('Count')
ax2.set_title('B  Clip Duration Distribution')
ax2.legend()

# Panel C: fault breakdown by genre
ax3 = fig.add_subplot(gs[2])
fault_pivot = meta.pivot_table(index='genre', columns='fault', aggfunc='size', fill_value=0)
fault_pivot = fault_pivot.reindex(GENRES)
fault_cols = [c for c in ['clean','corrupt','duplicate','mislabeled'] if c in fault_pivot.columns]
fault_pivot[fault_cols].plot(
    kind='bar', ax=ax3, stacked=True,
    color=['#4CAF50','#F44336','#FF9800','#9C27B0'],
    edgecolor='white', linewidth=0.5
)
ax3.set_xticklabels(GENRES, rotation=45, ha='right')
ax3.set_ylabel('Number of clips')
ax3.set_title('C  Fault Audit per Genre')
ax3.legend(loc='upper right', fontsize=9)

fig.suptitle('Figure 1 — GTZAN Dataset Composition & Fault Audit', fontweight='bold')
savefig('01_dataset_composition', fig)

In [ ]:
# ── Cell 6: Figure 02 — Representative spectrogram panel ─────────────────────
# One mel spectrogram per genre — 2x5 grid
SR = 22050
N_MELS = 128
HOP = 512

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for i, genre in enumerate(GENRES):
    # Pick first clean clip
    clips = meta[(meta.genre == genre) & (meta.fault == 'clean')].head(1)
    if clips.empty:
        continue
    path = clips.iloc[0]['filepath']
    y, sr = librosa.load(path, sr=SR, duration=30.0)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP)
    S_db = librosa.power_to_db(S, ref=np.max)
    img = librosa.display.specshow(
        S_db, sr=sr, hop_length=HOP, x_axis='time', y_axis='mel',
        ax=axes[i], cmap='magma'
    )
    axes[i].set_title(genre.capitalize(), fontweight='bold', color=GENRE_COLORS[genre])
    axes[i].set_xlabel('')
    axes[i].set_ylabel('')

# Add shared colorbar
fig.colorbar(img, ax=axes, format='%+2.0f dB', shrink=0.6, pad=0.02, label='Power (dB)')
fig.suptitle('Figure 2 — Mel Spectrogram Gallery (one clip per genre)', fontweight='bold')
savefig('01_spectrogram_gallery', fig)

In [ ]:
# ── Cell 7: Figure 03 — Waveform comparison Blues vs Jazz ────────────────────
target_pairs = [('blues', 'jazz'), ('metal', 'classical'), ('hiphop', 'reggae')]

fig, axes = plt.subplots(len(target_pairs), 2, figsize=(16, 4 * len(target_pairs)))

for row, (g1, g2) in enumerate(target_pairs):
    for col, genre in enumerate([g1, g2]):
        clip = meta[(meta.genre == genre) & (meta.fault == 'clean')].iloc[0]
        y, sr = librosa.load(clip.filepath, sr=SR, duration=30.0)
        times = np.linspace(0, len(y) / sr, len(y))
        ax = axes[row, col]
        ax.plot(times, y, color=GENRE_COLORS[genre], linewidth=0.4, alpha=0.85)
        ax.set_title(f'{genre.capitalize()}', fontweight='bold')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')
        ax.set_xlim(0, 30)

fig.suptitle('Figure 3 — Waveform Comparison Across Genre Pairs', fontweight='bold')
savefig('01_waveform_comparison', fig)

In [ ]:
# ── Cell 8: Feature extraction — MFCC, Chroma, Spectral Contrast, ZCR ────────
def extract_features(filepath):
    try:
        y, sr = librosa.load(filepath, sr=SR, duration=30.0)
    except Exception:
        return None

    mfcc       = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    chroma     = librosa.feature.chroma_stft(y=y, sr=sr)
    contrast   = librosa.feature.spectral_contrast(y=y, sr=sr)
    zcr        = librosa.feature.zero_crossing_rate(y)
    rms        = librosa.feature.rms(y=y)
    centroid   = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth  = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff    = librosa.feature.spectral_rolloff(y=y, sr=sr)
    tempo, _   = librosa.beat.beat_track(y=y, sr=sr)

    feats = {}
    for i, coeff in enumerate(mfcc):
        feats[f'mfcc_{i}_mean'] = coeff.mean()
        feats[f'mfcc_{i}_std']  = coeff.std()
    feats['chroma_mean']      = chroma.mean()
    feats['chroma_std']       = chroma.std()
    feats['contrast_mean']    = contrast.mean()
    feats['zcr_mean']         = zcr.mean()
    feats['zcr_std']          = zcr.std()
    feats['rms_mean']         = rms.mean()
    feats['rms_std']          = rms.std()
    feats['centroid_mean']    = centroid.mean()
    feats['bandwidth_mean']   = bandwidth.mean()
    feats['rolloff_mean']     = rolloff.mean()
    feats['tempo']            = float(tempo)
    return feats

all_feats = []
for _, row in tqdm(meta.iterrows(), total=len(meta), desc='Extracting features'):
    if row.fault == 'corrupt':
        continue
    f = extract_features(row.filepath)
    if f:
        f['genre']    = row.genre
        f['filename'] = row.filename
        f['fault']    = row.fault
        all_feats.append(f)

feat_df = pd.DataFrame(all_feats)
feat_df.to_csv(DATA / 'gtzan_features.csv', index=False)
print(f'Feature matrix: {feat_df.shape}')
feat_df.head(3)

In [ ]:
# ── Cell 9: Figure 04 — Feature distribution by genre ────────────────────────
feature_showcase = ['tempo', 'zcr_mean', 'centroid_mean', 'mfcc_0_mean', 'rms_mean', 'chroma_mean']
feature_labels   = ['Tempo (BPM)', 'ZCR Mean', 'Spectral Centroid', 'MFCC-0 Mean', 'RMS Energy', 'Chroma Mean']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, feat, label in zip(axes, feature_showcase, feature_labels):
    for genre in GENRES:
        vals = feat_df[feat_df.genre == genre][feat].dropna()
        ax.hist(vals, bins=20, alpha=0.55, label=genre,
                color=GENRE_COLORS[genre], edgecolor='none')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')

# Single shared legend
handles = [plt.Rectangle((0,0),1,1, color=GENRE_COLORS[g]) for g in GENRES]
fig.legend(handles, GENRES, loc='lower center', ncol=5, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), framealpha=0.9)

fig.suptitle('Figure 4 — Acoustic Feature Distributions by Genre', fontweight='bold')
savefig('01_feature_distributions', fig)

In [ ]:
# ── Cell 10: Figure 05 — Correlation heatmap of acoustic features ─────────────
num_cols = [c for c in feat_df.columns if feat_df[c].dtype in [np.float64, np.int64]
            and c not in ['fault']]
corr = feat_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    annot=False, linewidths=0.3,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.7}
)
ax.set_title('Figure 5 — Acoustic Feature Correlation Matrix (lower triangle)', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
savefig('01_feature_correlation', fig)

In [ ]:
# ── Cell 11: Figure 06 — Tempo boxplot per genre ─────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
data_by_genre = [feat_df[feat_df.genre == g]['tempo'].dropna().values for g in GENRES]
bp = ax.boxplot(data_by_genre, patch_artist=True, notch=True,
                medianprops=dict(color='black', linewidth=2))
for patch, genre in zip(bp['boxes'], GENRES):
    patch.set_facecolor(GENRE_COLORS[genre])
    patch.set_alpha(0.8)
ax.set_xticks(range(1, 11))
ax.set_xticklabels([g.capitalize() for g in GENRES], rotation=30, ha='right')
ax.set_ylabel('Estimated Tempo (BPM)')
ax.set_title('Figure 6 — Tempo Distribution by Genre (with notched box plots)', fontweight='bold')
savefig('01_tempo_boxplot', fig)

In [ ]:
# ── Cell 12: Figure 07 — MFCC heatmap per genre (mean across clips) ──────────
mfcc_cols_mean = [f'mfcc_{i}_mean' for i in range(20)]
genre_mfcc = feat_df.groupby('genre')[mfcc_cols_mean].mean().reindex(GENRES)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    genre_mfcc.T, ax=ax,
    cmap='coolwarm', center=0,
    xticklabels=[g.capitalize() for g in GENRES],
    yticklabels=[f'MFCC-{i}' for i in range(20)],
    cbar_kws={'label': 'Mean Coefficient Value'},
    annot=True, fmt='.1f', annot_kws={'fontsize': 7}
)
ax.set_title('Figure 7 — Mean MFCC Coefficients per Genre', fontweight='bold')
ax.set_xlabel('')
savefig('01_mfcc_heatmap', fig)

In [ ]:
# ── Cell 13: Clean split — fault-aware ───────────────────────────────────────
from sklearn.model_selection import StratifiedShuffleSplit

clean_df = feat_df[feat_df.fault == 'clean'].copy().reset_index(drop=True)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in sss.split(clean_df, clean_df['genre']):
    train_df = clean_df.iloc[train_idx]
    test_df  = clean_df.iloc[test_idx]

train_df.to_csv(DATA / 'train_features.csv', index=False)
test_df.to_csv(DATA  / 'test_features.csv',  index=False)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')
print('\nTrain genre distribution:')
print(train_df['genre'].value_counts())
print('\nAll outputs saved to ../data/')